New calibration script using the satellite observations for era5 calibration. ERA5 estimates > 0 on the GrIS are being adjusted to 0. Implemented parallel processing using GitHub Copilot. 

In [ ]:
import numpy as np
import rasterio as rio
from rasterio.plot import show
import pandas as pd
from pathlib import Path
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
icemask = rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/Masks/Icemask.tif')
landmask = rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/Masks/Landmask.tif')
lst_example = rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/BatchExport_LST/2000/GEMLST_MODIS_20000101.tif')

lst_data = lst_example.read(1)
lst_transform = lst_example.transform
lst_crs = lst_example.crs
lst_bounds = lst_example.bounds
lst_width = lst_example.width
lst_height = lst_example.height

# ADD NEW COEFFICIENTS, UPLOAD TO ASSETS AND UPDATE THE TABLE IMPORT
coef_era5 = pd.read_csv('/home/jovyan/work/AVOCA/GEM/Development/Data/Coefficients/coefficients_era5_from_sat.csv')
coef_land = coef_era5[coef_era5['area']=='land']
coef_ice = coef_era5[coef_era5['area']=='ice']

   n_obs_training  coef_before  intercept_before  R-squarred_before  \
0      1341318746        0.889             1.068              0.837   
1      7313007329        1.009            -0.662              0.885   

   rmse_before  n_obs_test  coef_calibrated  intercept_calibrated  \
0         5.65   574855486                1                -0.003   
1         4.31  3134150638                1                -0.001   

   R-squarred_after  rmse_after  area  
0             0.837        4.90  land  
1             0.885        4.23   ice  
n_obs_training            int64
coef_before             float64
intercept_before        float64
R-squarred_before       float64
rmse_before             float64
n_obs_test                int64
coef_calibrated           int64
intercept_calibrated    float64
R-squarred_after        float64
rmse_after              float64
area                     object
dtype: object


In [ ]:
# ERA5 Data: 
imfolder = "./GL1000m_reproj/"
imfiles = sorted(Path(imfolder).glob("*.tif"))

# CORRUPTED FILES TO MISSING
#	t2m_elvcorr_2001_d201 until t2m_elvcorr_2001_d210
#	t2m_elvcorr_2001_d339 until t2m_elvcorr_2001_d349
#	t2m_elvcorr_2019_d160

# markers (adjust if you need case-insensitive match or different substrings)
start_marker = "t2m_1000m_2000_d001.tif" # OBS D11
end_marker = "t2m_1000m_2010_d365.tif"   # OBS LEAP YEARS

# find first index containing the start marker and last index containing the end marker
start_idx = next((i for i, p in enumerate(imfiles) if start_marker in p.name), None)
end_idx = next((i for i, p in enumerate(imfiles) if end_marker in p.name), None)

if start_idx is None:
    raise FileNotFoundError(f"No file containing '{start_marker}' found under {imfolder}")
if end_idx is None:
    raise FileNotFoundError(f"No file containing '{end_marker}' found under {imfolder}")
if end_idx < start_idx:
    raise ValueError(f"End file '{end_marker}' appears before start file '{start_marker}' in sorted file order")

# slice inclusive range
imfiles = imfiles[start_idx:end_idx + 1]

In [ ]:
# Keep this low if disk is slow. Set to 1 for original sequential behavior.
N_WORKERS = max(10, (os.cpu_count() or 1) - 1)

landmask_arr = landmask.read(1)
icemask_arr = icemask.read(1)


def _print_progress(done, total, width=30):
    if total <= 0:
        return
    frac = done / total
    filled = int(width * frac)
    bar = '#' * filled + '-' * (width - filled)
    sys.stdout.write(f"\rProgress [{bar}] {done}/{total} ({frac * 100:5.1f}%)")
    sys.stdout.flush()


def _calibrate_one(imfile):
    with rio.open(imfile) as img:
        img = img.read(1).astype(np.float32)  # Keep original calibration logic.
        era5_land = img * coef_land['coef_before'].values[0] + coef_land['intercept_before'].values[0]
        era5_ice = img * coef_ice['coef_before'].values[0] + coef_ice['intercept_before'].values[0]
        era5_ice_min0ice = np.where(era5_ice > 0, 0, era5_ice)

        era5_calibrated = np.where(landmask_arr == 1, era5_land, np.where(icemask_arr == 1, era5_ice_min0ice, np.nan))

    output_file = '/home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL1000mCalSAT/t2m_1000m_cal_FILENAME.tif'
    output_file = output_file.replace("FILENAME", imfile.stem.split('_', 2)[2])
    with rio.open(output_file, 'w', driver='GTiff', height=lst_height, width=lst_width, count=1, dtype=np.float32, crs=lst_crs, transform=lst_transform) as dst:
        dst.write(era5_calibrated, 1)

    return f'File {imfile} calibrated'

In [ ]:
if N_WORKERS <= 1:
    done = 0
    total = len(imfiles)
    _print_progress(done, total)
    for imfile in imfiles:
        print('\n' + _calibrate_one(imfile))
        done += 1
        _print_progress(done, total)
    print()
else:
    done = 0
    total = len(imfiles)
    _print_progress(done, total)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = [ex.submit(_calibrate_one, imfile) for imfile in imfiles]
        for fut in as_completed(futures):
            print('\n' + fut.result())
            done += 1
            _print_progress(done, total)
    print()

print('ALL FILES DONE')

-0.662
